# ARC-v0.6.1 — Sealed HotpotQA Cross-Dataset Replication

## Hotfix rationale

ARC-v0.6 assumed FEVER-style qrels already contained `corpus-row`.
HotpotQA actually stores:

- `source/hotpotqa/qrels/dev.tsv`
- columns: `query-id`, `corpus-id`, `score`

and the corpus row mapping is stored in:

- `stage1/corpus_ids.sqlite`

ARC-v0.6.1 adds a deterministic adapter:

\[
\text{corpus-id}\rightarrow\text{SQLite corpus row}
\]

before evaluation.

`test.tsv` and every `stage3/test_*` artifact are explicitly prohibited.

## Protocol integrity note

Before this hotfix, only an artifact/schema audit was performed. That audit displayed the
first five DEV qrels rows, but **no HotpotQA retrieval outcome, trajectory metric, intervention
effect, or endpoint was observed**.

Therefore:
- H1–H4 are unchanged;
- feedback settings are unchanged;
- retrieval settings are unchanged;
- statistical tests and pass rules are unchanged;
- no HotpotQA outcome-dependent tuning is allowed.

The repaired protocol records this schema-audit event explicitly.

## Frozen cross-dataset hypotheses

### H1
\[
E[\mathrm{slope}_t D_q(t)]>0
\]

### H2
\[
E[\mathrm{slope}_t(D_C(t)-D_C(0))]>0
\]

### H3
\[
E[\mathrm{slope}_t|nDCG@10_{SQ8}(t)-nDCG@10_{PQ32}(t)|]>0
\]

### H4
\[
E[nDCG@10_B-nDCG@10_A]>0
\]

with

\[
A=PQ32\ search\rightarrow PQ32\ feedback
\]

and

\[
B=PQ32\ search\rightarrow SQ8\ feedback.
\]

All four endpoints must pass query-level bootstrap, sign-flip randomization, and joint Holm
correction for a strong cross-dataset replication.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, os, gc, sys, subprocess

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 20260816
TOP_RETRIEVE = 100
TOP_K = 10
MAX_ROUNDS = 4

NLIST = 4096
NPROBE = 64
PQ_M = 32
NBITS = 8

TRAIN_DOCS = 200_000
ADD_BATCH = 10_000

BOOTSTRAP_SAMPLES = 20_000
RANDOMIZATION_SAMPLES = 20_000
RESAMPLE_CHUNK = 250

HOT_ROOT = Path(
    "/content/drive/MyDrive/"
    "hc-rars-external-confirmation-hotpotqa-5m-v1"
)

FEVER_CONFIRM_ROOT = Path(
    "/content/drive/MyDrive/rag-pq-checkpoints/"
    "arc-v0/sealed-fever-dev-confirmation-v05"
)

ARC_ROOT = Path(
    "/content/drive/MyDrive/rag-pq-checkpoints/arc-v0"
)

CACHE_ROOT = Path(
    "/content/drive/MyDrive/rag-pq-checkpoints/arc-index-cache"
)
CACHE_ROOT.mkdir(parents=True, exist_ok=True)

OUT_ROOT = ARC_ROOT / "sealed-hotpotqa-replication-v06"
OUT_ROOT.mkdir(parents=True, exist_ok=True)

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S")
OUT = OUT_ROOT / RUN_ID
OUT.mkdir(parents=True, exist_ok=False)

print("HotpotQA root:", HOT_ROOT)
print("Output:", OUT)


In [ ]:
def run(cmd):
    print("$", " ".join(map(str,cmd)))
    subprocess.run(list(map(str,cmd)), check=True)

run([
    sys.executable, "-m", "pip", "install", "-q",
    "faiss-cpu==1.12.0", "psutil", "pyarrow"
])

import faiss, psutil

PROCESS = psutil.Process(os.getpid())

def ram_status(label=""):
    vm = psutil.virtual_memory()
    rss = PROCESS.memory_info().rss / 1024**3
    print(
        f"[RAM] {label:<30} "
        f"RSS={rss:6.2f} GB | "
        f"available={vm.available/1024**3:6.2f} GB | "
        f"used={vm.percent:5.1f}%"
    )

print("Faiss:", faiss.__version__)
ram_status("startup")


## 1. Recover the sealed FEVER configuration — no HotpotQA tuning


In [ ]:
fever_runs = sorted(
    [
        p for p in FEVER_CONFIRM_ROOT.glob("*")
        if p.is_dir()
        and (p/"report.json").is_file()
        and (p/"sealed_protocol.json").is_file()
    ],
    key=lambda p:p.stat().st_mtime,
)

if not fever_runs:
    raise FileNotFoundError(
        "No completed ARC-v0.5 sealed FEVER confirmation found."
    )

FEVER_RUN = fever_runs[-1]

with open(FEVER_RUN/"report.json","r",encoding="utf-8") as f:
    fever_report = json.load(f)

with open(FEVER_RUN/"sealed_protocol.json","r",encoding="utf-8") as f:
    fever_protocol = json.load(f)

if not fever_report.get("all_primary_endpoints_pass", False):
    raise RuntimeError(
        "Latest FEVER sealed confirmation did not pass all primary endpoints."
    )

selected_configs = fever_report["feedback_configs"]

print("Source FEVER sealed run:", FEVER_RUN)
print("Source protocol SHA:", fever_report["protocol_sha256"])
print("Frozen feedback configs:")
for c in selected_configs:
    print(c)


## 2. Use the audited HotpotQA artifacts deterministically

The artifact audit established the exact inputs:

- corpus: `stage1/corpus_embeddings.float16.memmap`
- corpus-ID mapping: `stage1/corpus_ids.sqlite`
- queries: `stage1/query_embeddings.float32.npy`
- query IDs: `stage1/query_ids.utf8.txt`
- official split membership: `stage1/official_split_manifest.json`
- confirmation relevance: `source/hotpotqa/qrels/dev.tsv`

No `test.tsv` or `stage3/test_*` file is opened.


In [ ]:
QUERY_EMB = HOT_ROOT / "stage1/query_embeddings.float32.npy"
QUERY_IDS = HOT_ROOT / "stage1/query_ids.utf8.txt"
CORPUS_EMB = HOT_ROOT / "stage1/corpus_embeddings.float16.memmap"
CORPUS_ID_DB = HOT_ROOT / "stage1/corpus_ids.sqlite"
SPLIT_MANIFEST = HOT_ROOT / "stage1/official_split_manifest.json"
QRELS_PATH = HOT_ROOT / "source/hotpotqa/qrels/dev.tsv"

required = [
    QUERY_EMB,
    QUERY_IDS,
    CORPUS_EMB,
    CORPUS_ID_DB,
    SPLIT_MANIFEST,
    QRELS_PATH,
]

missing = [str(p) for p in required if not p.is_file()]
if missing:
    raise FileNotFoundError("Missing required HotpotQA artifacts:\n- " + "\n- ".join(missing))

# Hard safety boundary.
for p in required:
    if "test" in p.name.lower():
        raise RuntimeError(f"Prohibited TEST artifact selected: {p}")

print("QUERY_EMB   :", QUERY_EMB)
print("QUERY_IDS   :", QUERY_IDS)
print("CORPUS_EMB  :", CORPUS_EMB)
print("CORPUS_ID_DB:", CORPUS_ID_DB)
print("SPLIT       :", SPLIT_MANIFEST)
print("DEV QRELS   :", QRELS_PATH)


## 3. Validate shapes, official DEV membership, and SQLite schema


In [ ]:
query_embeddings = np.load(QUERY_EMB, mmap_mode="r")
assert query_embeddings.ndim == 2

q_count, DIM = query_embeddings.shape

with open(QUERY_IDS, "r", encoding="utf-8") as f:
    query_ids = [x.strip() for x in f if x.strip()]

assert len(query_ids) == q_count
assert len(set(query_ids)) == len(query_ids)

bytes_per_row = DIM * np.dtype(np.float16).itemsize
nbytes = CORPUS_EMB.stat().st_size

assert nbytes % bytes_per_row == 0, (
    nbytes,
    DIM,
    bytes_per_row,
)

n_docs = nbytes // bytes_per_row

docs = np.memmap(
    CORPUS_EMB,
    dtype=np.float16,
    mode="r",
    shape=(n_docs, DIM),
)

with open(SPLIT_MANIFEST, "r", encoding="utf-8") as f:
    hot_split = json.load(f)

dev_ids_official = [str(x).strip() for x in hot_split["dev_query_ids"]]
test_ids_official = [str(x).strip() for x in hot_split["test_query_ids"]]

assert len(dev_ids_official) == 5447
assert hot_split["test_qrels_relevance_values_accessed"] is False
assert hot_split["test_retrieval_performed"] is False
assert hot_split["test_outcomes_observed"] is False

print("queries:", query_embeddings.shape, query_embeddings.dtype)
print("corpus :", docs.shape, docs.dtype)
print("official DEV:", len(dev_ids_official))
print("official TEST membership only:", len(test_ids_official))

# Introspect SQLite without assuming a table name.
with sqlite3.connect(str(CORPUS_ID_DB)) as con:
    tables = [
        r[0]
        for r in con.execute(
            "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name"
        ).fetchall()
    ]

    print("SQLite tables:", tables)

    mapping_table = None
    id_col = None
    row_col = None

    for table in tables:
        cols = [
            r[1]
            for r in con.execute(f'PRAGMA table_info("{table}")').fetchall()
        ]

        print(" ", table, cols)

        lower = {c.lower(): c for c in cols}

        possible_id = None
        for name in ["doc_id", "corpus_id", "corpus-id", "id"]:
            if name in lower:
                possible_id = lower[name]
                break

        possible_row = None
        for name in ["row_id", "corpus_row", "corpus-row", "row"]:
            if name in lower:
                possible_row = lower[name]
                break

        if possible_id is not None and possible_row is not None:
            mapping_table = table
            id_col = possible_id
            row_col = possible_row
            break

if mapping_table is None:
    raise RuntimeError(
        "Could not find a SQLite table containing both corpus/document ID and row ID."
    )

print("Selected mapping:", mapping_table, id_col, "->", row_col)


## 4. Seal the repaired cross-dataset protocol

The protocol is sealed **before any HotpotQA retrieval outcome is computed**.

For transparency, it records that the schema-repair audit previously displayed five DEV qrels
rows. This did not expose any retrieval effectiveness or H1–H4 outcome.


In [ ]:
protocol = {
    "schema_version": 2,
    "status": "SEALED_AFTER_SCHEMA_AUDIT_BEFORE_HOTPOTQA_RETRIEVAL",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "source_fever_v05_run": str(FEVER_RUN),
    "source_fever_protocol_sha256": fever_report["protocol_sha256"],
    "dataset": "HotpotQA",
    "query_embeddings": str(QUERY_EMB),
    "query_ids": str(QUERY_IDS),
    "corpus_embeddings": str(CORPUS_EMB),
    "corpus_id_database": str(CORPUS_ID_DB),
    "official_split_manifest": str(SPLIT_MANIFEST),
    "qrels": str(QRELS_PATH),
    "query_count_total": int(q_count),
    "official_dev_query_count": int(len(dev_ids_official)),
    "corpus_rows": int(n_docs),
    "dimension": int(DIM),
    "feedback_configs": selected_configs,
    "top_retrieve": TOP_RETRIEVE,
    "top_k": TOP_K,
    "feedback_rounds": MAX_ROUNDS,
    "low_fidelity": {
        "type": "IVF-PQ",
        "nlist": NLIST,
        "nprobe": NPROBE,
        "M": PQ_M,
        "nbits": NBITS,
    },
    "high_fidelity": {
        "type": "IVF-SQ8",
        "nlist": NLIST,
        "nprobe": NPROBE,
    },
    "primary_endpoints": [
        "H1_query_divergence_slope",
        "H2_candidate_increment_slope",
        "H3_abs_utility_gap_slope",
        "H4_B_minus_A",
    ],
    "primary_unit": "query",
    "method_pooling": "average within query across frozen FEVER-derived feedback configs",
    "bootstrap_samples": BOOTSTRAP_SAMPLES,
    "randomization_samples": RANDOMIZATION_SAMPLES,
    "multiple_testing": "Holm jointly across H1-H4",
    "endpoint_pass_rule": "mean>0 and CI_low>0 and Holm_p<0.05",
    "full_replication_rule": "all H1-H4 pass",
    "hotpotqa_tuning_allowed": False,
    "test_qrels_allowed": False,
    "schema_audit_before_this_protocol": {
        "performed": True,
        "reason": "repair qrels input adapter",
        "dev_qrels_rows_displayed": 5,
        "retrieval_outcomes_observed": False,
        "trajectory_endpoints_observed": False,
        "parameter_changes_after_outcomes": False,
    },
}

canonical = json.dumps(
    protocol,
    sort_keys=True,
    ensure_ascii=False,
    separators=(",", ":"),
).encode("utf-8")

protocol_sha256 = hashlib.sha256(canonical).hexdigest()
protocol["protocol_sha256"] = protocol_sha256

PROTOCOL_PATH = OUT / "sealed_protocol.json"
PROTOCOL_PATH.write_text(
    json.dumps(protocol, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

(OUT / "SEALED_BEFORE_HOTPOTQA_RETRIEVAL.txt").write_text(
    "SEALED_AFTER_SCHEMA_AUDIT\n"
    f"protocol_sha256={protocol_sha256}\n",
    encoding="utf-8",
)

print("SEALED:", PROTOCOL_PATH)
print("SHA-256:", protocol_sha256)


## 5. Convert HotpotQA DEV qrels `corpus-id → corpus-row`

Only `source/hotpotqa/qrels/dev.tsv` is read.

Every qrels query must belong to the official DEV membership. Corpus IDs are mapped through the
pre-existing SQLite corpus mapping. The adapter fails closed if any positive qrels document
cannot be mapped.


In [ ]:
# Explicit TSV parse; never read test.tsv.
assert QRELS_PATH.name == "dev.tsv"
assert "test" not in str(QRELS_PATH).lower()

qrels_raw = pd.read_csv(
    QRELS_PATH,
    sep="\t",
)

required_cols = {"query-id", "corpus-id", "score"}
assert required_cols.issubset(qrels_raw.columns)

qrels_raw["query-id"] = qrels_raw["query-id"].astype(str)
qrels_raw["corpus-id"] = qrels_raw["corpus-id"].astype(str)

official_dev_set = set(dev_ids_official)

unexpected_queries = sorted(
    set(qrels_raw["query-id"]) - official_dev_set
)

if unexpected_queries:
    raise RuntimeError(
        "DEV qrels contain queries outside official DEV membership: "
        + repr(unexpected_queries[:10])
    )

# Keep official DEV order, but require that relevance exists.
qrels_query_set = set(qrels_raw["query-id"])
confirmation_ids = [
    q for q in dev_ids_official
    if q in qrels_query_set
]

if len(confirmation_ids) != len(dev_ids_official):
    missing_qrels = [
        q for q in dev_ids_official
        if q not in qrels_query_set
    ]
    raise RuntimeError(
        f"Official DEV queries missing from dev.tsv: {len(missing_qrels)} "
        + repr(missing_qrels[:10])
    )

query_row = {qid: i for i, qid in enumerate(query_ids)}

missing_query_embeddings = [
    q for q in confirmation_ids
    if q not in query_row
]

if missing_query_embeddings:
    raise RuntimeError(
        f"DEV queries missing embeddings: {len(missing_query_embeddings)} "
        + repr(missing_query_embeddings[:10])
    )

confirmation_rows = np.array(
    [query_row[q] for q in confirmation_ids],
    dtype=np.int64,
)

# Map only corpus IDs actually used by positive DEV qrels.
positive = qrels_raw[qrels_raw["score"] > 0].copy()
unique_corpus_ids = positive["corpus-id"].drop_duplicates().tolist()

row_map = {}

with sqlite3.connect(str(CORPUS_ID_DB)) as con:
    sql = (
        f'SELECT "{row_col}" '
        f'FROM "{mapping_table}" '
        f'WHERE CAST("{id_col}" AS TEXT) = ?'
    )

    cur = con.cursor()

    for i, cid in enumerate(unique_corpus_ids):
        result = cur.execute(sql, (str(cid),)).fetchone()

        if result is not None:
            row_map[str(cid)] = int(result[0])

        if (i + 1) % 5000 == 0:
            print(f"mapped {i+1:,}/{len(unique_corpus_ids):,}")

missing_corpus = [
    cid for cid in unique_corpus_ids
    if str(cid) not in row_map
]

if missing_corpus:
    raise RuntimeError(
        f"Positive DEV qrels corpus IDs missing from SQLite: {len(missing_corpus)} "
        + repr(missing_corpus[:20])
    )

positive["corpus-row"] = (
    positive["corpus-id"]
    .map(row_map)
    .astype(np.int64)
)

assert positive["corpus-row"].between(0, n_docs - 1).all()

qrels = {}

for qid, g in positive.groupby("query-id"):
    qrels[qid] = set(
        g["corpus-row"].astype(np.int64).tolist()
    )

assert len(qrels) == len(confirmation_ids)

print("Confirmation DEV queries:", len(confirmation_ids))
print("Positive qrels rows       :", len(positive))
print("Unique relevant corpus IDs:", len(unique_corpus_ids))
print("All corpus IDs mapped     :", len(row_map))
print("TEST qrels accessed       : False")


## 6. Build/load memory-safe HotpotQA PQ32 and SQ8 indexes


In [ ]:
DATASET_TAG=f"hotpotqa-{n_docs}d-{DIM}dim"

PQ32_PATH=CACHE_ROOT/(
    f"{DATASET_TAG}-ivfpq-nlist{NLIST}-m{PQ_M}-"
    f"nbits{NBITS}-seed{SEED}.faiss"
)

SQ8_PATH=CACHE_ROOT/(
    f"{DATASET_TAG}-ivfsq8-nlist{NLIST}-seed{SEED}.faiss"
)

def build_index(kind,path):
    if path.is_file():
        print("Loading cached:",path)
        idx=faiss.read_index(str(path))
        idx.nprobe=NPROBE
        return idx

    rng=np.random.default_rng(SEED)
    train_ids=rng.choice(
        n_docs,
        size=min(TRAIN_DOCS,n_docs),
        replace=False,
    )

    train_x=np.ascontiguousarray(
        np.asarray(docs[train_ids],dtype=np.float32)
    )

    quantizer=faiss.IndexFlatIP(DIM)

    if kind=="pq32":
        idx=faiss.IndexIVFPQ(
            quantizer,DIM,NLIST,PQ_M,NBITS,
            faiss.METRIC_INNER_PRODUCT,
        )
    elif kind=="sq8":
        idx=faiss.IndexIVFScalarQuantizer(
            quantizer,DIM,NLIST,
            faiss.ScalarQuantizer.QT_8bit,
            faiss.METRIC_INNER_PRODUCT,
        )
    else:
        raise ValueError(kind)

    print("Training",kind)
    idx.train(train_x)

    del train_x
    gc.collect()

    for start in range(0,n_docs,ADD_BATCH):
        end=min(start+ADD_BATCH,n_docs)

        xb=np.asarray(
            docs[start:end],
            dtype=np.float32,
        )

        idx.add(
            np.ascontiguousarray(xb)
        )

        del xb

        if end % 250_000 < ADD_BATCH or end==n_docs:
            print(
                kind,
                f"{end:,}/{n_docs:,}",
                f"{100*end/n_docs:.1f}%"
            )
            ram_status(f"{kind} add")

    assert idx.ntotal==n_docs
    idx.nprobe=NPROBE

    faiss.write_index(idx,str(path))
    print("Saved:",path)

    return idx

# Build sequentially and release.
idx=build_index("pq32",PQ32_PATH)
del idx
gc.collect()

idx=build_index("sq8",SQ8_PATH)
del idx
gc.collect()

print("Index preparation complete.")


## 7. Shared retrieval helpers


In [ ]:
def normalize_rows(x):
    x=np.asarray(x,np.float32)
    n=np.linalg.norm(x,axis=1,keepdims=True)
    return x/np.maximum(n,1e-12)

def evaluate_batch(qids,ranked_ids,k=10):
    recall=np.empty(len(qids),np.float32)
    mrr=np.empty(len(qids),np.float32)
    ndcg=np.empty(len(qids),np.float32)

    discounts=1.0/np.log2(np.arange(2,k+2))

    for i,qid in enumerate(qids):
        relset=qrels.get(qid,set())
        ranked=ranked_ids[i,:k]

        hits=np.array(
            [1.0 if int(d) in relset else 0.0 for d in ranked],
            dtype=np.float32,
        )

        recall[i]=hits.sum()/max(len(relset),1)

        pos=np.flatnonzero(hits)
        mrr[i]=1.0/(pos[0]+1) if len(pos) else 0.0

        dcg=float((hits*discounts).sum())
        ideal=min(len(relset),k)
        idcg=float(discounts[:ideal].sum()) if ideal else 0.0
        ndcg[i]=dcg/idcg if idcg>0 else 0.0

    return recall,mrr,ndcg

def jaccard_rows(a,b):
    vals=np.empty(len(a),np.float32)

    for i in range(len(a)):
        A=set(map(int,a[i]))
        B=set(map(int,b[i]))
        vals[i]=len(A&B)/max(len(A|B),1)

    return vals

def cosine_distance_rows(a,b):
    a=normalize_rows(a)
    b=normalize_rows(b)
    return 1.0-np.sum(a*b,axis=1)

def feedback_matrix(ids,scores,config,batch_size=256):
    k=int(config["k"])
    out=np.empty((len(ids),DIM),np.float32)

    for start in range(0,len(ids),batch_size):
        end=min(start+batch_size,len(ids))

        dids=np.asarray(
            ids[start:end,:k],
            dtype=np.int64,
        )

        x=np.asarray(
            docs[dids],
            dtype=np.float32,
        )

        if config["method"]=="mean":
            f=x.mean(axis=1)

        elif config["method"]=="softmax":
            temp=float(config["temperature"])
            z=np.asarray(
                scores[start:end,:k],
                np.float64,
            )/temp
            z-=z.max(axis=1,keepdims=True)

            w=np.exp(np.clip(z,-60,60))
            w/=np.maximum(
                w.sum(axis=1,keepdims=True),
                1e-12,
            )

            f=(x*w[:,:,None]).sum(axis=1)

        else:
            raise ValueError(config["method"])

        f=f.astype(np.float32)
        f/=np.maximum(
            np.linalg.norm(f,axis=1,keepdims=True),
            1e-12,
        )

        out[start:end]=f

    return out

def anchored_update_matrix(q0,f,alpha):
    q=(
        (1.0-float(alpha))*q0
        + float(alpha)*f
    ).astype(np.float32)

    q/=np.maximum(
        np.linalg.norm(q,axis=1,keepdims=True),
        1e-12,
    )

    return q

def config_key(c):
    temp="none" if c.get("temperature") is None else str(c["temperature"]).replace(".","p")
    return (
        f"{c['method']}"
        f"-k{int(c['k'])}"
        f"-a{str(c['alpha']).replace('.','p')}"
        f"-t{temp}"
    )

Q0=normalize_rows(
    np.asarray(
        query_embeddings[confirmation_rows],
        dtype=np.float32,
    )
)


## 8. Synchronized HotpotQA trajectories


In [ ]:
def run_self_feedback(index_path,condition,config):
    index=faiss.read_index(str(index_path))
    index.nprobe=NPROBE

    q0=Q0
    qt=q0.copy()
    states=[]

    for t in range(MAX_ROUNDS+1):
        print(condition,config_key(config),"iteration",t)

        scores,ids=index.search(
            np.ascontiguousarray(qt,np.float32),
            TOP_RETRIEVE,
        )

        recall,mrr,ndcg=evaluate_batch(
            confirmation_ids,ids,TOP_K
        )

        states.append({
            "q":qt.copy(),
            "ids":ids.copy(),
            "scores":scores.copy(),
            "recall":recall,
            "mrr":mrr,
            "ndcg":ndcg,
        })

        if t<MAX_ROUNDS:
            f=feedback_matrix(ids,scores,config)
            qt=anchored_update_matrix(
                q0,f,config["alpha"]
            )

    del index
    gc.collect()
    return states

state_cache={}

for config in selected_configs:
    ck=config_key(config)

    for condition,path in [
        ("ivfpq32",PQ32_PATH),
        ("ivfsq8",SQ8_PATH),
    ]:
        cache_path=OUT/f"states-{condition}-{ck}.npz"

        if cache_path.is_file():
            z=np.load(cache_path)
            states=[]

            for t in range(MAX_ROUNDS+1):
                states.append({
                    "q":z[f"q_{t}"],
                    "ids":z[f"ids_{t}"],
                    "scores":z[f"scores_{t}"],
                    "recall":z[f"recall_{t}"],
                    "mrr":z[f"mrr_{t}"],
                    "ndcg":z[f"ndcg_{t}"],
                })

        else:
            states=run_self_feedback(
                path,condition,config
            )

            payload={}

            for t,s in enumerate(states):
                for field in [
                    "q","ids","scores",
                    "recall","mrr","ndcg"
                ]:
                    payload[f"{field}_{t}"]=s[field]

            np.savez_compressed(
                cache_path,
                **payload,
            )

        state_cache[(condition,ck)]=states

print("HotpotQA synchronized trajectories complete.")


## 9. H1–H3 endpoints


In [ ]:
trajectory_rows=[]

for config in selected_configs:
    ck=config_key(config)

    A=state_cache[("ivfpq32",ck)]
    B=state_cache[("ivfsq8",ck)]

    dc0=None

    for t in range(MAX_ROUNDS+1):
        dq=cosine_distance_rows(
            A[t]["q"],
            B[t]["q"],
        )

        dc=1.0-jaccard_rows(
            A[t]["ids"],
            B[t]["ids"],
        )

        if t==0:
            dc0=dc.copy()

        dc_inc=dc-dc0

        ugap=(
            B[t]["ndcg"]
            - A[t]["ndcg"]
        )

        augap=np.abs(ugap)

        for i,qid in enumerate(confirmation_ids):
            trajectory_rows.append({
                "query_id":qid,
                "method":config["method"],
                "config_key":ck,
                "iteration":t,
                "query_divergence":float(dq[i]),
                "candidate_divergence_increment":float(dc_inc[i]),
                "abs_utility_gap":float(augap[i]),
                "pq32_ndcg":float(A[t]["ndcg"][i]),
                "sq8_ndcg":float(B[t]["ndcg"][i]),
            })

trajectory_df=pd.DataFrame(trajectory_rows)

def linear_slope(g,metric):
    g=g.sort_values("iteration")
    return float(
        np.polyfit(
            g["iteration"].to_numpy(np.float64),
            g[metric].to_numpy(np.float64),
            1,
        )[0]
    )

method_rows=[]

for (qid,method,ck),g in trajectory_df.groupby(
    ["query_id","method","config_key"]
):
    method_rows.append({
        "query_id":qid,
        "method":method,
        "config_key":ck,
        "H1_query_divergence_slope":
            linear_slope(g,"query_divergence"),
        "H2_candidate_increment_slope":
            linear_slope(g,"candidate_divergence_increment"),
        "H3_abs_utility_gap_slope":
            linear_slope(g,"abs_utility_gap"),
    })

method_slopes=pd.DataFrame(method_rows)

query_slopes=(
    method_slopes
    .groupby("query_id",as_index=False)
    [[
        "H1_query_divergence_slope",
        "H2_candidate_increment_slope",
        "H3_abs_utility_gap_slope",
    ]]
    .mean()
)

display(query_slopes.describe())


## 10. H4 intervention


In [ ]:
def run_intervention(config):
    pq32=faiss.read_index(str(PQ32_PATH))
    sq8=faiss.read_index(str(SQ8_PATH))

    pq32.nprobe=NPROBE
    sq8.nprobe=NPROBE

    q0=Q0
    qA=q0.copy()
    qB=q0.copy()

    rows=[]

    for t in range(MAX_ROUNDS+1):
        print("intervention",config_key(config),"iteration",t)

        sA,idA=pq32.search(
            np.ascontiguousarray(qA,np.float32),
            TOP_RETRIEVE,
        )

        sB,idB=pq32.search(
            np.ascontiguousarray(qB,np.float32),
            TOP_RETRIEVE,
        )

        sBfb,idBfb=sq8.search(
            np.ascontiguousarray(qB,np.float32),
            TOP_RETRIEVE,
        )

        _,_,nA=evaluate_batch(
            confirmation_ids,idA,TOP_K
        )

        _,_,nB=evaluate_batch(
            confirmation_ids,idB,TOP_K
        )

        if t==MAX_ROUNDS:
            for i,qid in enumerate(confirmation_ids):
                rows.append({
                    "query_id":qid,
                    "method":config["method"],
                    "config_key":config_key(config),
                    "A_ndcg":float(nA[i]),
                    "B_ndcg":float(nB[i]),
                    "H4_B_minus_A":float(nB[i]-nA[i]),
                })

        if t<MAX_ROUNDS:
            fA=feedback_matrix(idA,sA,config)
            fB=feedback_matrix(idBfb,sBfb,config)

            qA=anchored_update_matrix(
                q0,fA,config["alpha"]
            )

            qB=anchored_update_matrix(
                q0,fB,config["alpha"]
            )

    del pq32,sq8
    gc.collect()

    return pd.DataFrame(rows)

intervention=pd.concat(
    [run_intervention(c) for c in selected_configs],
    ignore_index=True,
)

query_intervention=(
    intervention
    .groupby("query_id",as_index=False)["H4_B_minus_A"]
    .mean()
)

display(query_intervention.describe())


## 11. Confirmatory statistics


In [ ]:
def bootstrap_mean(values,samples=BOOTSTRAP_SAMPLES,seed=0):
    x=np.asarray(values,np.float64)
    x=x[np.isfinite(x)]

    n=len(x)
    rng=np.random.default_rng(seed)
    draws=[]
    remain=samples

    while remain>0:
        b=min(RESAMPLE_CHUNK,remain)
        idx=rng.integers(
            0,n,size=(b,n),dtype=np.int32
        )
        draws.append(
            x[idx].mean(axis=1)
        )
        remain-=b

    draws=np.concatenate(draws)

    return {
        "n":int(n),
        "mean":float(x.mean()),
        "median":float(np.median(x)),
        "ci_low":float(np.quantile(draws,.025)),
        "ci_high":float(np.quantile(draws,.975)),
    }

def sign_flip(values,samples=RANDOMIZATION_SAMPLES,seed=0):
    x=np.asarray(values,np.float64)
    x=x[np.isfinite(x)]

    n=len(x)
    obs=float(x.mean())
    rng=np.random.default_rng(seed)

    exceed=0
    done=0

    while done<samples:
        b=min(RESAMPLE_CHUNK,samples-done)

        signs=rng.integers(
            0,2,size=(b,n),dtype=np.int8
        ).astype(np.float32)*2-1

        stats=(signs*x[None,:]).mean(axis=1)

        exceed+=int(np.sum(stats>=obs))
        done+=b

    return (exceed+1)/(samples+1)

def holm_adjust(pvalues):
    p=np.asarray(pvalues,np.float64)
    m=len(p)

    order=np.argsort(p)
    adjusted=np.empty(m,np.float64)

    running=0.0

    for rank,idx in enumerate(order):
        running=max(
            running,
            (m-rank)*p[idx],
        )
        adjusted[idx]=min(running,1.0)

    return adjusted

endpoint_values={
    "H1_query_divergence_slope":
        query_slopes["H1_query_divergence_slope"].to_numpy(),
    "H2_candidate_increment_slope":
        query_slopes["H2_candidate_increment_slope"].to_numpy(),
    "H3_abs_utility_gap_slope":
        query_slopes["H3_abs_utility_gap_slope"].to_numpy(),
    "H4_B_minus_A":
        query_intervention["H4_B_minus_A"].to_numpy(),
}

rows=[]

for i,(name,x) in enumerate(endpoint_values.items()):
    b=bootstrap_mean(
        x,
        seed=SEED+100+i,
    )
    p=sign_flip(
        x,
        seed=SEED+200+i,
    )

    rows.append({
        "endpoint":name,
        **b,
        "randomization_p":p,
        "fraction_positive":float(np.mean(x>0)),
        "fraction_zero":float(np.mean(np.isclose(x,0,atol=1e-12))),
        "fraction_negative":float(np.mean(x<0)),
    })

confirm_df=pd.DataFrame(rows)

confirm_df["holm_p"]=holm_adjust(
    confirm_df["randomization_p"]
)

confirm_df["pass"]=(
    (confirm_df["mean"]>0)
    & (confirm_df["ci_low"]>0)
    & (confirm_df["holm_p"]<0.05)
)

display(confirm_df)


## 12. Cross-dataset verdict


In [ ]:
all_pass=bool(confirm_df["pass"].all())

print("=== ARC-v0.6 SEALED HOTPOTQA CROSS-DATASET REPLICATION ===")
print("Protocol SHA-256:",protocol_sha256)
print("Confirmation queries:",len(confirmation_ids))
print()

display(
    confirm_df[
        [
            "endpoint","mean","ci_low","ci_high",
            "randomization_p","holm_p",
            "fraction_positive","pass"
        ]
    ]
)

if all_pass:
    decision=(
        "CROSS-DATASET REPLICATION PASS: all four FEVER-derived approximation-feedback "
        "amplification hypotheses replicate on HotpotQA without HotpotQA-based tuning. "
        "Proceed to a third-domain replication and practical selective-fidelity mitigation."
    )
else:
    failed=confirm_df.loc[
        ~confirm_df["pass"],
        "endpoint"
    ].tolist()

    decision=(
        "CROSS-DATASET REPLICATION PARTIAL/FAIL: failed endpoints: "
        + ", ".join(failed)
        + ". Keep the FEVER result but do not claim generality until the failure is diagnosed."
    )

print("DECISION:",decision)


## 13. Save evidence


In [ ]:
trajectory_df.to_parquet(
    OUT/"hotpotqa_paired_trajectories.parquet",
    index=False,
)

method_slopes.to_csv(
    OUT/"hotpotqa_method_slopes.csv",
    index=False,
)

query_slopes.to_csv(
    OUT/"hotpotqa_query_slopes.csv",
    index=False,
)

intervention.to_csv(
    OUT/"hotpotqa_intervention.csv",
    index=False,
)

query_intervention.to_csv(
    OUT/"hotpotqa_query_intervention.csv",
    index=False,
)

confirm_df.to_csv(
    OUT/"confirmatory_endpoints.csv",
    index=False,
)

report={
    "status":"SEALED_HOTPOTQA_CROSS_DATASET_REPLICATION_V061_COMPLETE",
    "protocol_sha256":protocol_sha256,
    "source_fever_protocol_sha256":fever_report["protocol_sha256"],
    "confirmation_query_count":len(confirmation_ids),
    "qrels_path":str(QRELS_PATH),
    "feedback_configs":selected_configs,
    "primary_endpoints":confirm_df.to_dict(orient="records"),
    "all_primary_endpoints_pass":all_pass,
    "decision":decision,
    "hotpotqa_tuning_performed":False,
    "schema_adapter_repair_only":True,
    "test_qrels_accessed":False,
    "completed_at_utc":datetime.now(timezone.utc).isoformat(),
}

REPORT_PATH=OUT/"report.json"

REPORT_PATH.write_text(
    json.dumps(
        report,
        indent=2,
        ensure_ascii=False,
        default=float,
    ),
    encoding="utf-8",
)

report_sha=hashlib.sha256(
    REPORT_PATH.read_bytes()
).hexdigest()

(OUT/"FINAL_REPORT_SHA256.txt").write_text(
    report_sha+"\n",
    encoding="utf-8",
)

print("Saved:",OUT)
print("Report:",REPORT_PATH)
print("Report SHA-256:",report_sha)
